In [ ]:
import os

import pandas as pd
from tqdm import tqdm

from guild.tools.scrapping import download_pdb_files

tqdm.pandas()
pd.set_option("mode.chained_assignment", None)

# Get all GPCR PDB files
These include downlading of PDB files in the notebook folder.

In [ ]:
downloaded_pdbs_folder = "../data/pdbs"
publication_data_folder = "../../guild/support/publication_data"

os.makedirs(downloaded_pdbs_folder, exist_ok=True)
os.makedirs(publication_data_folder, exist_ok=True)

In [ ]:
curated_data = pd.read_csv(
    "../../guild/support/GPCRs_curated_dataset.tsv",
    sep="\t",
    header=0,
)
usable_curated_data = curated_data[curated_data["Usable"] == "YES"]
usable_curated_data["PDB_ID"] = usable_curated_data["PDB_ID"].str.lower()
usable_curated_pdbs = usable_curated_data["PDB_ID"].tolist()
f"Number of GPCR PDB files to download: {len(usable_curated_pdbs)}"

In [ ]:
failed_pdbs = download_pdb_files(
    usable_curated_pdbs, download_dir=downloaded_pdbs_folder
)

# Subset to successful PDBs only

In [ ]:
downloaded_curated_data = usable_curated_data[
    ~usable_curated_data["PDB_ID"].isin(failed_pdbs)
]
f"Number of GPCR PDB files successfully downloaded: {len(downloaded_curated_data['PDB_ID'].tolist())}"

## Add UniProt mappings

In [ ]:
gpcrdb_mapping = pd.read_csv(
    "../../guild/support/uniprot_gpcrdb_mapping.txt",
    sep=" ",
)
final_data = pd.merge(
    gpcrdb_mapping,
    usable_curated_data,
    left_on="pdb_id",
    right_on="PDB_ID",
    how="inner",
)
final_data.head()

In [ ]:
final_data.to_csv(
    f"{publication_data_folder}/gpcrdb_protein_data.csv",
    index=False,
)
len(final_data)

# Subsetting GPCR to limited proteins

Since each GPCR faimly has multiple proteins, the idea here is to sample "n" proteins from each family to be the representative set for publication purposes.

In [ ]:
shuffled_data = final_data.sample(frac=1).reset_index(drop=True)
len(shuffled_data)

In [ ]:
# Removing families with less than 3 proteins
familes_removed = []

gpcr_grouped_counts = shuffled_data.groupby("gpcrdb_id").size()
for gpcr_id, gpcr_count in gpcr_grouped_counts.items():
    if gpcr_count < 3:
        familes_removed.append(gpcr_id)

cleaned_final_data = shuffled_data[~shuffled_data["gpcrdb_id"].isin(familes_removed)]
len(cleaned_final_data)

In [ ]:
gpcr_grouped = cleaned_final_data.groupby("gpcrdb_id")
len(gpcr_grouped)

## Set of 3 proteins per family

In [ ]:
gpcr_df_sample_three = gpcr_grouped.head(3).reset_index(drop=True)
len(gpcr_df_sample_three)

In [ ]:
gpcr_df_sample_three.to_csv(
    f"{publication_data_folder}/gpcrdb_protein_data_three.csv",
    index=False,
)

## Set of 1 proteins per family

In [ ]:
gpcr_df_sample_one = gpcr_grouped.head(1).reset_index(drop=True)
len(gpcr_df_sample_one)

In [ ]:
gpcr_df_sample_one.to_csv(
    f"{publication_data_folder}/gpcrdb_protein_data_one.csv",
    index=False,
)